In [3]:
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

# Access API key
groq_api_key = os.getenv("GROQ_API_KEY")

# Optional safety check
if groq_api_key is None:
    raise ValueError("GROQ_API_KEY not found")

print("API Key Loaded ✅")

API Key Loaded ✅


In [ ]:
# !pip install -q langchain-groq langchain-community langchain-core requests duckduckgo-search

In [5]:
from langchain_groq import ChatGroq   # ✅ correct
from langchain_core.tools import tool
import requests

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [15]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=4d1d8ae207a8c845a52df8a67bf3623e&query={city}'

  response = requests.get(url)

  return response.json()

In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [10]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", 
     """You are a helpful AI agent that can reason step-by-step and use tools.

You must follow this format:

Thought: think about what to do
Action: the tool to use
Action Input: the input to the tool
Observation: result from the tool

Repeat the above steps if needed.

When you have the final answer:
Final Answer: give the final result clearly

Rules:
- Use tools only when necessary
- Do not make up answers
- Always follow the format strictly
"""),
    
    ("human", "{input}")
])

In [11]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an AI agent.

- Think step by step
- Use tools when needed
- Return final answer clearly
"""),
    
    ("human", "{input}")
])

In [18]:
# ---------------------------------------------------------------
# 🔥 Modern ReAct Agent (Fully Working)
# ---------------------------------------------------------------

import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

load_dotenv()

# ---------------------------------------------------------------
# 🤖 LLM
# ---------------------------------------------------------------

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# ---------------------------------------------------------------
# 🛠️ Define Tools
# ---------------------------------------------------------------

@tool
def search_tool(query: str) -> str:
    """Search the internet for general information."""
    return f"Search results for: {query}"

@tool
def get_weather_data(city: str) -> str:
    """Get weather information for a city."""
    return f"The weather in {city} is 25°C and sunny."


# ---------------------------------------------------------------
# 🧠 System Prompt
# ---------------------------------------------------------------

system_prompt = """
You are an AI agent.

Think step by step.
Use tools when needed.
Return final answer clearly.
"""

# ---------------------------------------------------------------
# 🧠 Bind Tools
# ---------------------------------------------------------------

llm_with_tools = llm.bind_tools([search_tool, get_weather_data])

# ---------------------------------------------------------------
# 🚀 Input
# ---------------------------------------------------------------

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="What is the weather in Delhi?")
]

# ---------------------------------------------------------------
# 🔁 Agent Loop
# ---------------------------------------------------------------

while True:

    response = llm_with_tools.invoke(messages)

    if not response.tool_calls:
        print("\n✅ Final Answer:")
        print(response.content)
        break

    tool_call = response.tool_calls[0]
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    print("\n🧠 Thought: Using tool")
    print(f"🔧 Tool: {tool_name}")
    print(f"📥 Input: {tool_args}")

    # Execute tool
    if tool_name == "search_tool":
        result = search_tool.invoke(tool_args)

    elif tool_name == "get_weather_data":
        result = get_weather_data.invoke(tool_args)

    print(f"📤 Output: {result}")

    # Send result back
    messages.append(response)
    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    )


🧠 Thought: Using tool
🔧 Tool: get_weather_data
📥 Input: {'city': 'Delhi'}
📤 Output: The weather in Delhi is 25°C and sunny.

🧠 Thought: Using tool
🔧 Tool: search_tool
📥 Input: {'query': 'Delhi weather today'}
📤 Output: Search results for: Delhi weather today

✅ Final Answer:
The current weather in Delhi is 25°C and sunny.


In [19]:
# ---------------------------------------------------------------
# 🔥 Modern ReAct Agent (No deprecated APIs)
# ---------------------------------------------------------------

from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

# ---------------------------------------------------------------
# 🧠 System Prompt (ReAct style)
# ---------------------------------------------------------------

system_prompt = """
You are an AI agent.

Follow this pattern:
Thought → Action → Observation → Repeat

Use tools when needed.
Return final answer clearly.
"""

# ---------------------------------------------------------------
# 🧠 Bind Tools
# ---------------------------------------------------------------

llm_with_tools = llm.bind_tools([search_tool, get_weather_data])

# ---------------------------------------------------------------
# 🚀 Input
# ---------------------------------------------------------------

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="What is the weather in Delhi?")
]

# ---------------------------------------------------------------
# 🔁 ReAct Loop
# ---------------------------------------------------------------

while True:

    response = llm_with_tools.invoke(messages)

    # If no tool call → done
    if not response.tool_calls:
        print("\n✅ Final Answer:")
        print(response.content)
        break

    # Tool call
    tool_call = response.tool_calls[0]
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    print("\n🧠 Thought: Using tool")
    print(f"🔧 Tool: {tool_name}")
    print(f"📥 Input: {tool_args}")

    # Execute correct tool
    if tool_name == "search_tool":
        result = search_tool.invoke(tool_args)

    elif tool_name == "get_weather_data":
        result = get_weather_data.invoke(tool_args)

    print(f"📤 Output: {result}")

    # Add to conversation
    messages.append(response)
    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    )


🧠 Thought: Using tool
🔧 Tool: get_weather_data
📥 Input: {'city': 'Delhi'}
📤 Output: The weather in Delhi is 25°C and sunny.

🧠 Thought: Using tool
🔧 Tool: get_weather_data
📥 Input: {'city': 'Delhi'}
📤 Output: The weather in Delhi is 25°C and sunny.

🧠 Thought: Using tool
🔧 Tool: get_weather_data
📥 Input: {'city': 'Delhi'}
📤 Output: The weather in Delhi is 25°C and sunny.

✅ Final Answer:
The weather in Delhi is 25°C and sunny.


In [21]:
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="Find the capital of Madhya Pradesh, then find its current weather condition")
]

while True:

    response = llm_with_tools.invoke(messages)

    if not response.tool_calls:
        print("\n✅ Final Answer:")
        print(response.content)
        break

    tool_call = response.tool_calls[0]
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    print("\n🧠 Thought: Using tool")
    print(f"🔧 Tool: {tool_name}")
    print(f"📥 Input: {tool_args}")

    # Execute tools
    if tool_name == "search_tool":
        result = search_tool.invoke(tool_args)

    elif tool_name == "get_weather_data":
        result = get_weather_data.invoke(tool_args)

    print(f"📤 Output: {result}")

    messages.append(response)
    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    )


🧠 Thought: Using tool
🔧 Tool: get_weather_data
📥 Input: {'city': 'Bhopal'}
📤 Output: The weather in Bhopal is 25°C and sunny.

🧠 Thought: Using tool
🔧 Tool: get_weather_data
📥 Input: {'city': 'Bhopal'}
📤 Output: The weather in Bhopal is 25°C and sunny.

✅ Final Answer:
The capital of Madhya Pradesh is Bhopal.


In [ ]:
response['output']

'The capital of Madhya Pradesh is Bhopal, and the current weather condition in Bhopal is partly cloudy with a temperature of 40°C.'